# Day 3 · Hands-on 1 — Survival analysis: censoring, Kaplan–Meier, log-rank and Cox
**About 40 min · work in pairs** (one drives, one reads, predicts and questions — swap halfway)

Many biomedical outcomes are **times to an event**: death, relapse, infection, a tumour reaching a given size. Some patients haven't had the event when the study ends or when they leave it. Their time is **censored**: we know only that it is *longer than* what we observed. Survival methods use that partial information instead of throwing it away or pretending it is complete.

> **Two AI roles — keep them apart**
> - 🧮 **Gemini in Colab = calculator.** Ask it to write, change, fix or explain *code*, including new simulations.
> - 🎓 **Claude / ChatGPT (other tab) = tutor.** Use it to explain ideas, quiz you and critique your own work. Ask for its reasoning, and check one claim yourself in the notebook.

**How this notebook works:** run cells from top to bottom. Each code cell has one line marked `# 🔧 CHANGE THIS` — the only thing you need to edit. Lines starting with 📋 go in the shared sheet. Lines starting with 🎯 appear only for simulated data, where we know the true answer.

In [ ]:
# @title ▶️ Setup: run this cell first (no changes needed) { display-mode: "form" }
# Every analysis below uses ROLES (time, event, group, ...), never hard-coded column names.
# To add a dataset: write a loader function and add one entry to DATASETS.
%pip install -q lifelines
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test, proportional_hazard_test
from lifelines.plotting import add_at_risk_counts
from lifelines.utils import median_survival_times
from lifelines.datasets import load_gbsg2, load_lung
warnings.filterwarnings("ignore")

def _gbsg2():
    d = load_gbsg2()
    for c in ["tgrade", "menostat", "horTh"]:
        d[c] = d[c].astype(str)
    d["hormone_therapy"] = (d["horTh"] == "yes").astype(int)
    d["postmenopausal"] = (d["menostat"] == "Post").astype(int)
    d["grade_II"] = (d["tgrade"] == "II").astype(int)
    d["grade_III"] = (d["tgrade"] == "III").astype(int)
    return d

def _lung():
    d = load_lung()
    d.columns = [c.replace(".", "_") for c in d.columns]
    d["sex"] = d["sex"].map({1: "male", 2: "female"})
    d["female"] = (d["sex"] == "female").astype(int)
    return d

def _simulated(seed=2026, n=400):
    rng = np.random.default_rng(seed)
    arm = rng.choice(["control", "drug"], size=n)
    drug = (arm == "drug").astype(int)
    age = rng.normal(58, 8, n) + 10 * drug                    # the drug was given to older patients
    hazard = np.log(2) / 24 * np.exp(np.log(0.6) * drug + np.log(1.04) * (age - 60))
    true_time = rng.exponential(1 / hazard)
    followup = np.minimum(rng.uniform(12, 60, n), rng.exponential(200, n))   # end of study or drop-out
    return pd.DataFrame(dict(months=np.minimum(true_time, followup).round(2),
                             died=(true_time <= followup).astype(int),
                             arm=arm, drug=drug, age=age.round(1)))

def _my_data():
    # Upload a CSV to Colab (folder icon on the left) and edit the roles in DATASETS["my_data"]
    return pd.read_csv("my_data.csv")

DATASETS = {
    "gbsg2": dict(load=_gbsg2,
        about="GBSG2 breast cancer trial: 686 patients, outcome = recurrence-free survival",
        time="time", event="cens", unit="days", horizon=1825, horizon_label="5 years",
        group="horTh", group_ref="no", group_dummy="hormone_therapy", group_label="Hormone therapy",
        covariates=["age", "postmenopausal", "tsize", "pnodes", "grade_II", "grade_III"]),
    "lung": dict(load=_lung,
        about="NCCTG advanced lung cancer study: 228 patients, outcome = death",
        time="time", event="status", unit="days", horizon=365, horizon_label="1 year",
        group="sex", group_ref="male", group_dummy="female", group_label="Sex",
        covariates=["age", "ph_ecog", "ph_karno"]),
    "simulated": dict(load=_simulated,
        about="Simulated trial-like study: 400 patients, true effects known",
        time="months", event="died", unit="months", horizon=36, horizon_label="3 years",
        group="arm", group_ref="control", group_dummy="drug", group_label="Treatment arm",
        covariates=["age"],
        truth={"cox": "HR drug vs control = 0.60 (adjusted for age); HR per year of age = 1.04. "
                      "Drug patients are ~10 years older, so the crude HR is pulled towards 1 (about 0.6 × 1.04^10 ≈ 0.89).",
               "median": "true median survival for a 60-year-old control patient = 24 months"}),
    "my_data": dict(load=_my_data,
        about="Your own CSV: edit these roles to match your column names",
        time="my_time", event="my_event_1_if_happened", unit="days", horizon=365, horizon_label="1 year",
        group="my_group", group_ref="my_reference_level", group_dummy="my_group_0_1", group_label="My group",
        covariates=["my_covariate"]),
}

def load_dataset(name):
    R = dict(DATASETS[name])
    d = R.pop("load")()
    d[R["group"]] = d[R["group"]].astype(str)
    R.setdefault("truth", {})
    print(f"✅ {R['about']}  ({len(d)} rows)")
    print(f"   time: {R['time']} ({R['unit']})   event: {R['event']} (1 = event, 0 = censored)   group: {R['group']}")
    print(f"   possible adjustment variables: {R['covariates']}")
    return d, R

def surv_at(km, t):
    """Survival probability and 95% CI at time t from a fitted KaplanMeierFitter."""
    s = km.survival_function_at_times(t).iloc[0]
    ci = km.confidence_interval_survival_function_
    row = ci[ci.index <= t].iloc[-1]
    return s, row.iloc[0], row.iloc[1]

def fmt_median(km):
    m = km.median_survival_time_
    if np.isinf(m):
        return "not reached"
    lo, hi = median_survival_times(km.confidence_interval_).iloc[0]
    return f"{m:.0f} (95% CI {lo:.0f} to {'not reached' if np.isinf(hi) else f'{hi:.0f}'})"

def hr_table(cph):
    s = cph.summary
    t = pd.DataFrame({"HR": s["exp(coef)"], "CI low": s["exp(coef) lower 95%"],
                      "CI high": s["exp(coef) upper 95%"], "p-value": s["p"].map(lambda p: f"{p:.2g}")})
    return t.round(3)

def show_truth(R, key):
    if key in R["truth"]:
        print(f"🎯 Truth used to simulate: {R['truth'][key]}")

print("✅ Setup complete")

## A. Why censoring matters (a simulation where we know the truth)
### 🧭 Concept
We simulate 200 patients whose **true** survival times we know: the true median is **24 months**. Patients are recruited during the first half of the study, so when the study stops each one has been followed for a different length of time. Anyone still alive at that point is censored.

We then estimate the median survival three ways:
1. **Pretend censored times are deaths** (use every observed time as if it were the event time).
2. **Drop the censored patients** and use only those who died.
3. **Kaplan–Meier**, which uses censored patients for as long as they were observed.

✍️ **Predict first:** with a short study, will approaches 1 and 2 overestimate or underestimate the median? Which one gets worse as the study gets shorter?

In [ ]:
# 🔧 CHANGE THIS: how long the study runs, in months (try 12, 24, 48, 120)
study_length = 48

rng = np.random.default_rng(1)
n = 200
true_time = rng.exponential(24 / np.log(2), n)        # true median = 24 months
followup = rng.uniform(study_length / 2, study_length, n)   # recruited during the first half of the study
observed = np.minimum(true_time, followup)
event = (true_time <= followup).astype(int)

km = KaplanMeierFitter(label="Kaplan–Meier").fit(observed, event)
naive = KaplanMeierFitter(label="1. censored treated as deaths").fit(observed, np.ones(n))
dropped = KaplanMeierFitter(label="2. censored patients dropped").fit(observed[event == 1])

fig, ax = plt.subplots(figsize=(7, 4.5))
t = np.linspace(0, 120, 300)
ax.plot(t, np.exp(-np.log(2) / 24 * t), "k--", lw=2, label="True survival curve")
naive.plot_survival_function(ax=ax, ci_show=False, color="tab:orange")
dropped.plot_survival_function(ax=ax, ci_show=False, color="tab:gray")
km.plot_survival_function(ax=ax, ci_show=True, color="tab:purple")
ax.axvline(study_length, color="red", lw=1, ls=":"); ax.axhline(0.5, color="gray", lw=0.5)
ax.set_xlim(0, max(60, study_length * 1.1)); ax.set_ylim(0, 1.02)
ax.set_xlabel("Months"); ax.set_ylabel("Proportion still alive"); ax.legend(fontsize=8); plt.show()

print(f"Censored: {1 - event.mean():.0%} of patients")
print(f"True median                       = 24.0 months")
print(f"1. Censored treated as deaths     = {np.median(observed):.1f} months")
print(f"2. Censored patients dropped      = {np.median(observed[event == 1]):.1f} months")
print(f"3. Kaplan–Meier median            = {fmt_median(km)} months")
print(f"📋 study_length = {study_length}: censored {1 - event.mean():.0%}, naive median {np.median(observed):.1f}, KM median {fmt_median(km)}")

### ✅ Concept check
With `study_length = 12`, the Kaplan–Meier median is "not reached". Is that a failure of the method?

<details><summary>Show answer</summary>

No: it is honest. Fewer than half the patients died during the observed follow-up, so the data cannot tell you when half of them will have died. Approaches 1 and 2 give you a number anyway, and it is badly wrong (far too short): both treat the censored patients' long survival as if it never happened. When the median is not reached, report survival at a fixed time point (e.g. "82% alive at 12 months") instead.
</details>

Kaplan–Meier makes one key assumption: **censoring is non-informative**. Patients who leave the study must have the same prospects as those who stay. If the sickest patients drop out (too ill to attend clinic), Kaplan–Meier is too optimistic.

## Choose a dataset
Everything below is written in terms of **roles** (time, event, group), so the same cells work on any dataset. Start with the one your facilitator names, then try `simulated`: it was generated with known effects (and a trap).

In [ ]:
# 🔧 CHANGE THIS: which dataset to use (all cells below adapt automatically)
dataset = "gbsg2"  # @param ["gbsg2", "lung", "simulated", "my_data"]

df, R = load_dataset(dataset)
T, E, G = R["time"], R["event"], R["group"]
print(f"\nEvents: {df[E].sum()}   Censored: {(1 - df[E]).sum()} ({1 - df[E].mean():.0%})")
df.head()

## B. Look at the raw data, then draw Kaplan–Meier curves
### 🧭 Concept
Each line below is one patient's follow-up. A dot means the event happened; an open circle means censored (still event-free when last seen). Notice that censored patients appear at **all** follow-up lengths, not only at the end.

In [ ]:
# 🔧 CHANGE THIS: how many patients to show
n_show = 30

sample = df.sample(n_show, random_state=1).sort_values(T).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(7, 0.18 * n_show + 1))
for i, r in sample.iterrows():
    ax.hlines(i, 0, r[T], color="gray", lw=1)
    ax.plot(r[T], i, "o", color="black" if r[E] == 1 else "white", markeredgecolor="black")
ax.set_xlabel(f"Follow-up time ({R['unit']})"); ax.set_yticks([]); ax.set_title("● event   ○ censored")
plt.show()

The **Kaplan–Meier** curve steps down at each event. At each step, the drop is (events at that time) ÷ (patients still **at risk**). Censored patients count in the "at risk" denominator until they leave, so they contribute everything we know about them.

✍️ **Predict first:** from the swimmer plot, which group do you expect to have better survival?

In [ ]:
# 🔧 CHANGE THIS: variable that defines the curves (any categorical column)
group = G

fig, ax = plt.subplots(figsize=(7, 5))
kms = []
for lvl, sub in df.dropna(subset=[group]).groupby(group):
    km = KaplanMeierFitter(label=f"{group} = {lvl}").fit(sub[T], sub[E])
    km.plot_survival_function(ax=ax, ci_show=True, show_censors=True, censor_styles={"ms": 4, "marker": "|"})
    kms.append(km)
add_at_risk_counts(*kms, ax=ax, rows_to_show=["At risk"])
ax.set_xlabel(f"Time ({R['unit']})"); ax.set_ylabel("Proportion event-free"); ax.set_ylim(0, 1.02)
plt.tight_layout(); plt.show()

h = R["horizon"]
for km in kms:
    s, lo, hi = surv_at(km, h)
    print(f"{km._label:28s} median = {fmt_median(km)} {R['unit']};  at {R['horizon_label']}: {s:.0%} (95% CI {lo:.0%}–{hi:.0%})")
    print(f"📋 {dataset} | {km._label} | median {fmt_median(km)} | {R['horizon_label']}: {s:.0%}")

### ✅ Concept check
Why do the confidence bands get wider towards the right of the plot?

<details><summary>Show answer</summary>

Fewer patients are still at risk (look at the numbers under the plot). Each event late in follow-up is a big fraction of a small denominator, so the curve becomes jumpy and uncertain. Don't over-interpret the tail of a Kaplan–Meier curve, and always show the numbers at risk.
</details>

## C. Comparing curves: the log-rank test
### 🧭 Concept
The **log-rank test** asks: at each event time, did the events fall in each group in proportion to how many patients were at risk there? It adds up observed minus expected events over the whole follow-up.

On Day 1 you saw what goes wrong if you run a t-test on the observed times instead. Here are both side by side.

✍️ **Predict first:** will the t-test on observed times give a larger or smaller p-value than the log-rank test?

In [ ]:
# 🔧 CHANGE THIS: variable that defines the groups to compare
group = G

d = df.dropna(subset=[group])
lr = multivariate_logrank_test(d[T], d[group], d[E])
print(f"Log-rank test: χ² = {lr.test_statistic:.2f}, df = {lr.degrees_of_freedom}, p = {lr.p_value:.2g}")

levels = sorted(d[group].unique())
if len(levels) == 2:
    a, b = (d.loc[d[group] == l, T] for l in levels)
    tt = stats.ttest_ind(a, b, equal_var=False)
    print(f"Welch t-test on observed times (ignores censoring!): p = {tt.pvalue:.2g}")
    print(f"   mean observed time: {levels[0]} = {a.mean():.0f}, {levels[1]} = {b.mean():.0f} {R['unit']}")
print(f"📋 {dataset} | {group} | log-rank p = {lr.p_value:.2g}")

Did the two p-values agree? Even when the t-test p-value is *smaller*, it is not a bonus: an observed time mixes **how long a patient survived** with **how long she was followed**. If one group happened to be recruited earlier (longer follow-up), its observed times are longer even with identical survival. The log-rank test only compares groups among patients who are still being followed.

### ✅ Concept check
The log-rank test gives every event time equal weight. When will it have little power, even though the two curves are clearly different?

<details><summary>Show answer</summary>

When the curves **cross**: e.g. a treatment with early toxicity but later benefit. Excess deaths early and fewer deaths late cancel out in the sum. Always look at the curves before trusting a log-rank p-value. The same issue affects the Cox model (section E).
</details>

## D. Cox regression and the hazard ratio
### 🧭 Concept
The **hazard** is the instantaneous event rate among patients who are still event-free at a given time. A **Cox model** assumes the hazards in two groups are proportional: one is a constant multiple of the other at every time point. That multiple is the **hazard ratio (HR)**.

- HR = 0.7: at any time, among patients still event-free, the event rate in the treated group is 70% of the control rate.
- It does **not** mean "30% fewer patients die" or "patients live 30% longer". Those are different quantities (section B's Kaplan–Meier estimates give those).

Like any regression, Cox lets you **adjust** for other variables. On Day 1 (HO3) you saw that the GBSG2 hormone therapy groups differ in age and menopausal status. Does adjusting change the HR?

✍️ **Predict first:** start with `adjust_for = []` (crude HR), then add variables one at a time. Will the HR for the group move towards 1, away from 1, or stay put?

In [ ]:
# 🔧 CHANGE THIS: variables to adjust for (start with [], then e.g. ["age"], then R["covariates"])
adjust_for = []

cols = [R["group_dummy"]] + adjust_for
d = df.dropna(subset=cols + [T, E])
cox_data = d[cols + [T, E]]
cph = CoxPHFitter().fit(cox_data, duration_col=T, event_col=E)
print(f"Cox model on {len(d)} patients, {int(d[E].sum())} events.  {R['group_dummy']} = 1 vs {R['group_ref']}\n")
display(hr_table(cph))
g = cph.summary.loc[R["group_dummy"]]
label = "crude" if not adjust_for else "adjusted for " + ", ".join(adjust_for)
print(f"📋 {dataset} | HR {R['group_dummy']} {label}: {g['exp(coef)']:.2f} (95% CI {g['exp(coef) lower 95%']:.2f}–{g['exp(coef) upper 95%']:.2f})")
show_truth(R, "cox")

### ✅ Concept check
In `simulated`, the crude HR for the drug is close to 1 but the age-adjusted HR is about 0.6. Which one answers "does the drug work?", and why did they disagree?

<details><summary>Show answer</summary>

The adjusted HR. Older patients were more likely to get the drug, and age itself raises the hazard (HR 1.04 per year, so about 1.5 for 10 years). In the crude comparison the drug's benefit is cancelled out by the drug group's older age: **confounding**, the same idea as Day 2. In a randomised trial the groups would be balanced on age by design, and crude and adjusted HRs would agree (adjusting would only make the estimate more precise).

In GBSG2 the therapy groups differ in menopausal status, yet adjusting barely moves the HR (0.69 → 0.70). A variable only confounds if it is linked to **both** the treatment and the outcome; once age is in the model, menopausal status adds little.
</details>

**From HR to something a patient understands.** An HR is relative. Translate it into absolute terms with the Kaplan–Meier estimates from section B: "at 5 years, 58% vs 44% were recurrence-free". The same HR means a big absolute difference when events are common and a tiny one when they are rare.

## E. ⭐ Optional: is the proportional-hazards assumption reasonable?
### 🧭 Concept
If hazards are proportional, curves of **log(−log S(t))** against log(time) are roughly **parallel**. Crossing or converging lines suggest the effect changes over time, and a single HR then averages over that change. The Schoenfeld residual test gives a p-value per variable (small p = evidence against proportional hazards). As with normality tests on Day 2, with large samples it flags trivial departures: look at the plot first.

In [ ]:
# 🔧 CHANGE THIS: variable to check (a categorical column for the plot)
group = G

fig, ax = plt.subplots(figsize=(6, 4))
for lvl, sub in df.dropna(subset=[group]).groupby(group):
    km = KaplanMeierFitter().fit(sub[T], sub[E])
    sf = km.survival_function_.iloc[:, 0]
    sf = sf[(sf > 0) & (sf < 1) & (sf.index > 0)]
    ax.step(np.log(sf.index), np.log(-np.log(sf)), where="post", label=f"{group} = {lvl}")
ax.set_xlabel("log(time)"); ax.set_ylabel("log(−log S(t))"); ax.legend(); plt.show()

print("Schoenfeld residual test for the model you fitted in section D:")
display(proportional_hazard_test(cph, cox_data, time_transform="rank").summary.round(3))

## 🤖 Go deeper with AI
Three ways to keep learning about survival analysis: now, if you finish early, or after the workshop.

**🎓 Explain and quiz me** (Claude / ChatGPT):
> *Explain censoring, Kaplan–Meier curves and the hazard ratio to a lab scientist. Then give me 4 sentences from papers that report hazard ratios and ask me which are correct.*

**🧪 See it in a simulation** (🧮 Gemini in Colab, in a new cell at the end of this notebook):
> *Simulate two groups with exponential survival times, a true hazard ratio of 0.7 and about 50% censoring. Fit a Cox model 500 times and plot the estimated hazard ratios. Then simulate curves that cross.*

**🔬 Apply it to my work** (Claude / ChatGPT; replace the brackets with your own details):
> *My outcome is [for example: days until a tumour reaches 1,000 mm³; some mice were culled early]. Is this time-to-event data? What is censored, and could the censoring be related to prognosis?*

✔ **Check one claim yourself:** The estimated hazard ratios should centre on 0.7.

## 💬 Discuss with your partner
- In your own work, which outcomes are really times to an event, even if you usually analyse them as yes/no at a fixed time (e.g. "relapsed within 1 year")? What do you lose by doing that?
- Can you think of a reason why patients in your field might drop out for reasons linked to their prognosis (informative censoring)?
- In the lung dataset, `wt_loss` was used on Day 2 as a yes/no outcome. Look at the variables: what other time-to-event question could you ask with these data? (You'll need this in the afternoon capstone.)